# `prep_tracking`: do ATD bruto ao tracking limpo

Este notebook apresenta o processamento implementado em `preprocessing/prep_tracking.py` e alguns exemplos dos arquivos Parquet já gerados. A organização segue os notebooks `starter`: carregamos os dados, explicamos o significado das colunas e inspecionamos frames e movimentos.

O arquivo ATD tem um registro por frame. O pré-processamento alinha os frames à linha do tempo do vídeo tático, converte coordenadas para metros, orienta os dois tempos e calcula variáveis de qualidade e movimento. Os resultados persistidos são `tracking_clean.parquet` e `slot_teams.parquet`.

## 1. Localizar e abrir os dados

Instale as dependências com `python -m pip install -r requirements-preprocessing.txt`. O PyArrow é necessário para ler os Parquet e Matplotlib para os gráficos.

Se os Parquet processados ainda não existirem, gere-os primeiro com `python -m preprocessing.prep_tracking` na raiz do repositório. O comando precisa dos assets ATD do jogo em `data/tactical` ou no caminho definido por `MTAH_DATA_DIR` (consulte `DATA.md`). O Parquet contém uma linha por `video_frame`; os 46 slots ficam em colunas `s01_*` a `s46_*`.

In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((path for path in (cwd, *cwd.parents) if (path / "games.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Não encontrei a raiz do repositório (games.py) nos diretórios acima do diretório atual.")
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import games

processed = Path(games.DATA_DIR) / "processed"
tracking_path = processed / "tracking_clean.parquet"
teams_path = processed / "slot_teams.parquet"
print("tracking:", tracking_path, "| existe:", tracking_path.is_file())
print("slots:   ", teams_path, "| existe:", teams_path.is_file())
missing = [path for path in (tracking_path, teams_path) if not path.is_file()]
if missing:
    raise FileNotFoundError("Arquivos processados ausentes: " + ", ".join(map(str, missing)) + '. Gere-os com `python -m preprocessing.prep_tracking` na raiz do repositório.')

tracking = pd.read_parquet(tracking_path)
slot_teams = pd.read_parquet(teams_path)
print(f"\n{len(tracking):,} frames; {tracking.shape[1]} colunas")
print(f"{len(slot_teams)} slots de tracking")

: 

### Exemplos rápidos de consulta

O backend PyArrow é declarado em `requirements-preprocessing.txt`. Estes exemplos mostram como selecionar um período, separar os slots por time e medir a cobertura da bola.

In [ ]:
# Leia somente as colunas necessárias quando quiser uma inspeção leve.
sample = pd.read_parquet(
    tracking_path, columns=["video_frame", "half", "ball_x", "ball_y", "ball_observed"],
    engine="pyarrow",
)
sample.query("half == 1").head()

In [ ]:
# Liste os slots de cada time a partir do mapeamento persistido.
home_slots = slot_teams.loc[slot_teams.is_home, "slot"].tolist()
away_slots = slot_teams.loc[~slot_teams.is_home, "slot"].tolist()
print("slots mandante:", home_slots)
print("slots visitante:", away_slots)

# Cobertura de bola apenas nos frames avaliados.
graded = tracking.loc[tracking.is_graded]
graded.ball_observed.mean()

## 2. O que `prep_tracking` faz

O fluxo principal tem duas etapas: `read_atd` transforma o texto ATD em arrays normalizados de jogadores e bola, e `prepare_tracking` converte esses arrays em tabelas frame-alinhadas. `build` conecta as duas etapas e grava os Parquet.

Na leitura, `frame_id` começa em 1 e vira `video_frame = frame_id - 1`. A preparação então:

1. converte posições normalizadas em coordenadas de campo (metros);
2. reflete cada tempo quando necessário para que o mandante ataque no sentido de x crescente;
3. remove saltos implausíveis da bola, interpola lacunas de até 12 frames e calcula velocidade/aceleração;
4. marca quebras nos slots, remove detecções isoladas e duplicadas e calcula velocidade dos jogadores;
5. agrega centroide, dispersão, velocidade média e contagem de jogadores rastreados por time.

Halftime e frames fora dos períodos rastreados permanecem desconhecidos (`NaN` nas variáveis correspondentes), em vez de receberem posições inventadas.

## 3. Contrato das tabelas geradas

Cada linha de `tracking_clean` é indexada pelo frame do vídeo tático. `half == 0` identifica frames sem tracking de jogo; `is_graded` indica que o frame pertence a um dos períodos avaliados. Para cada slot há posição, velocidade e indicador de quebra. `slot_teams` mantém o mapeamento estável do número do slot para mandante/visitante.

In [ ]:
print("Colunas de frame e bola:")
print([c for c in tracking.columns if c in [
    "video_frame", "half", "is_graded", "frame_in_half",
    "ball_x", "ball_y", "ball_x_raw", "ball_y_raw", "ball_observed",
    "ball_jump_removed", "ball_filled", "gap_length", "ball_speed"]])
print("\nColunas do primeiro slot:", [c for c in tracking.columns if c.startswith("s01_")])
slot_teams.head(40)

## 4. Exemplo das saídas: primeiros frames avaliados

O início do primeiro período está no `video_frame` 11185. A tabela abaixo mostra colunas do arquivo já processado, incluindo a bola original preservada para auditoria (`ball_*_raw`) e a posição após filtragem/interpolação (`ball_x`, `ball_y`).

In [ ]:
kickoff_1 = 11185
cols = ["video_frame", "half", "is_graded", "frame_in_half",
        "ball_x_raw", "ball_y_raw", "ball_x", "ball_y",
        "ball_observed", "ball_jump_removed", "ball_filled", "ball_speed",
        "s01_x", "s01_y", "s01_vx", "s01_vy", "s01_break"]
cols = [c for c in cols if c in tracking.columns]
tracking.loc[kickoff_1:kickoff_1 + 9, cols]

Os valores de posição estão em metros e as velocidades em m/s. `ball_observed=1` significa que a bola passou pelo filtro como observação válida. `ball_filled=1` marca uma posição interpolada; ela não deve ser confundida com uma medição direta.

## 5. Bola observada, removida e interpolada

O filtro de saltos conserva uma cadeia temporal plausível. Lacunas curtas são interpoladas; lacunas maiores permanecem ausentes. Veja uma janela de 12 segundos a partir do pontapé inicial. A visualização compara x bruto e x preparado, e deixa visível quais frames foram preenchidos.

In [ ]:
# Escolha automaticamente um trecho que contenha ao menos um frame interpolado.
filled_all = tracking["ball_filled"].fillna(False).astype(bool)
if not filled_all.any():
    raise ValueError("Não há frames interpolados em tracking; confira o pré-processamento.")

filled_idx = tracking.index[filled_all]
center = filled_idx[0]
radius = 2 * games.GAMES[games.GAME]["fps"]
window = tracking.loc[center - radius:center + radius].copy()
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(window["video_frame"], window["ball_x_raw"], ".", ms=3, alpha=.45, label="ATD orientado (bruto)")
ax.plot(window["video_frame"], window["ball_x"], lw=1.4, label="posição preparada")
filled = window["ball_filled"].fillna(False).astype(bool)
ax.scatter(window.loc[filled, "video_frame"], window.loc[filled, "ball_x"],
           s=24, marker="s", label="interpolado", zorder=3)
ax.set(xlabel="video_frame", ylabel="x da bola (m)", title=f"Bola: interpolação em torno do frame {center}")
ax.legend(); ax.grid(alpha=.2); plt.show()

print("Janela exibida:", int(window["video_frame"].min()), "a", int(window["video_frame"].max()))
print("Frames interpolados na janela:", int(filled.sum()))
print("Cobertura de bola observada no jogo:", f"{tracking['ball_observed'].mean():.1%}")
print("Frames removidos como salto:", int(tracking["ball_jump_removed"].sum()))
print("Frames preenchidos:", int(tracking["ball_filled"].sum()))


## 6. Exemplo de movimento de um slot

Os slots são identidades de tracking temporárias, e não números de camisa ou identidades de jogadores. O gráfico mostra o percurso de `s01` em uma janela curta. Quebras (`s01_break`) indicam descontinuidade; a velocidade não atravessa uma quebra ou uma lacuna.

In [ ]:
# Use o próprio kickoff, que tem um frame de câmera conhecido. Escolhe um slot
# com posição nesse frame, em vez de começar pelo primeiro aparecimento de s01.
slots = sorted({int(c[1:-2]) for c in tracking.columns
                if c.startswith("s") and c.endswith("_x")
                and c[1:-2].isdigit()})
slots_no_kickoff = [s for s in slots
                    if tracking.loc[kickoff_1, [f"s{s:02d}_x", f"s{s:02d}_y"]].notna().all()]
if not slots_no_kickoff:
    raise ValueError(f"Nenhum slot tem posição válida no kickoff (frame {kickoff_1}).")
slot = slots_no_kickoff[0]
slot_x, slot_y = f"s{slot:02d}_x", f"s{slot:02d}_y"
slot_vx, slot_vy, slot_break = f"s{slot:02d}_vx", f"s{slot:02d}_vy", f"s{slot:02d}_break"
first_frame = kickoff_1
print(f"Slot {slot:02d} no frame de kickoff: {first_frame}")

frames_slot = tracking.loc[first_frame:, ["video_frame", slot_x, slot_y, slot_vx, slot_vy, slot_break]]
frames_slot = frames_slot.dropna(subset=[slot_x, slot_y]).head(10)
display(frames_slot)

window = tracking.loc[first_frame:first_frame + 20 * games.GAMES[games.GAME]["fps"]]
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(window[slot_x], window[slot_y], "-o", ms=2, lw=1, alpha=.75)
breaks = window[slot_break].astype(bool)
ax.scatter(window.loc[breaks, slot_x], window.loc[breaks, slot_y],
           color="crimson", s=35, label="break")
ax.set(xlim=(0, games.GAMES[games.GAME]["pitch"][0]),
       ylim=(0, games.GAMES[games.GAME]["pitch"][1]),
       xlabel="x (m)", ylabel="y (m)", title=f"Trajetória do slot {slot:02d}")
ax.set_aspect("equal"); ax.legend(); ax.grid(alpha=.2); plt.show()

print("Time do slot:", "mandante" if bool(slot_teams.loc[slot_teams.slot == slot, "is_home"].iloc[0]) else "visitante")


In [ ]:
fps = games.GAMES[games.GAME]["fps"]

# Tempo do vídeo, contado desde o frame 0
inicio_video_s = first_frame / fps
fim_frame = int(frames_slot["video_frame"].iloc[-1])
fim_video_s = fim_frame / fps

# Tempo decorrido desde o kickoff
inicio_kickoff_s = (first_frame - kickoff_1) / fps
fim_kickoff_s = (fim_frame - kickoff_1) / fps
duracao_trajetoria_s = (fim_frame - first_frame + 1) / fps

def mmss(segundos):
    minutos = int(segundos // 60)
    segundos_restantes = int(segundos % 60)
    return f"{minutos:02d}:{segundos_restantes:02d}"

print(f"Trajetória do slot {slot:02d}:")
print(f"  Vídeo: {mmss(inicio_video_s)} – {mmss(fim_video_s)}")
print(f"  Desde o kickoff: {mmss(inicio_kickoff_s)} – {mmss(fim_kickoff_s)}")
print(f"  Duração observada (incluindo os dois frames): {duracao_trajetoria_s:.2f} s")
print(f"  Frames: {first_frame} – {fim_frame} (FPS: {fps})")

In [ ]:
from preprocessing import prep_tracking

print("Funções do fluxo:")
print("  read_atd       -> arrays (jogadores, bola, times)")
print("  prepare_tracking -> tracking_clean e slot_teams")
print("  build          -> lê os assets e grava os dois Parquet")
print("\nBlocos de frames por tempo:", games.half_video_blocks("BAR-ATH"))
print("Dimensões do campo (m):", games.GAMES[games.GAME]["pitch"])

## 7. Resumo dos períodos e estatísticas por time

O intervalo entre os dois tempos não representa jogo em andamento. Os agregados por time são úteis para descrever a estrutura coletiva sem atribuir uma identidade permanente a cada slot.

In [ ]:
tracking.groupby("half")["video_frame"].agg(["min", "max", "count"])

In [ ]:
summary_cols = ["n_tracked_home", "n_tracked_away", "centroid_home_x", "centroid_away_x",
                "mean_speed_home", "mean_speed_away", "spread_home_x", "spread_away_x"]
summary_cols = [c for c in summary_cols if c in tracking.columns]
tracking.loc[kickoff_1:kickoff_1 + 9, ["video_frame", "half"] + summary_cols]